# Course Benchmark: FD Acoustic 2D Homogeneous Animation

This notebook uses the same core setup as `fd_acoustic_2D_homogeneous.ipynb` from the Computational Geophysics short course: `nx=500`, `nz=500`, `nt=502`, and `idisp=5`.

This is intentionally much heavier than the 1D benchmark.

In [ ]:
import os
import sys
import time
from pathlib import Path

for candidate in [Path('/files/notebooks'), Path('/notebooks'), Path.cwd(), Path.cwd() / 'notebooks']:
    if candidate.exists():
        os.chdir(candidate)
        sys.path.insert(0, str(candidate))
        print(f'working_directory={Path.cwd()}')
        break

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from IPython.display import HTML, display
from progress_bar import ProgressBarHandler
from animation_2d import create_animation_homogeneous

plt.rcParams['animation.embed_limit'] = 500

In [ ]:
nx   = 500
nz   = nx
dx   = 1.
dz   = dx
c0   = 580.
isx  = 200
isz  = isx
irx  = 300
irz  = irx
nt   = 502
dt   = 0.0010
f0   = 25.
t0   = 2. / f0
op   = 3
idisp = 5

eps  = c0 * dt / dx
print('Source frequency =', f0, 'Hz')
print('Stability criterion =', eps)

p    = np.zeros((nz, nx))
pold = np.zeros((nz, nx))
pnew = np.zeros((nz, nx))
d2px = np.zeros((nz, nx))
d2pz = np.zeros((nz, nx))
c    = np.zeros((nz, nx)) + c0
x    = np.arange(nx) * dx
z    = np.arange(nz) * dz
seis = np.zeros(nt)

src  = np.zeros(nt + 1)
time_values = np.linspace(0 * dt, nt * dt, nt)
src  = -8. * (time_values - t0) * f0 * (np.exp(-1.0 * (4 * f0) ** 2 * (time_values - t0) ** 2))

In [ ]:
fig1 = plt.figure(figsize=(10, 6))
gs1  = gridspec.GridSpec(1, 2, width_ratios=[1, 1], hspace=0.3, wspace=0.3)
ax1  = plt.subplot(gs1[0])
ax1.plot(time_values, src)
ax1.set_title('Source Time Function')
ax1.set_xlim(time_values[0], time_values[-1])
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')

ax2  = plt.subplot(gs1[1])
spec = np.fft.fft(src)
freq = np.fft.fftfreq(spec.size, d=dt)
ax2.plot(np.abs(freq), np.abs(spec))
ax2.set_xlim(0, 250)
ax2.set_title('Source Spectrum')
ax2.set_xlabel('Frequency (Hz)')
ax2.yaxis.tick_right()
ax2.yaxis.set_label_position('right')
plt.show()

In [ ]:
G = time_values * 0.
r = np.sqrt((x[isx] - x[irx]) ** 2 + (z[isz] - z[irz]) ** 2)
for it in range(nt):
    if (time_values[it] - r / c0) >= 0:
        G[it] = (1. / (2 * np.pi * c0 ** 2)) * (1. / np.sqrt((time_values[it] ** 2) - (r ** 2 / (c0 ** 2))))
Gc = np.convolve(G, src * dt)[0:nt]
lim = Gc.max()

fig2 = plt.figure(figsize=(10, 6))
gs2 = gridspec.GridSpec(1, 2, width_ratios=[1, 1], hspace=0.3, wspace=0.3)
ax3 = plt.subplot(gs2[0])
leg1, = ax3.plot(isx, isz, 'r*', markersize=11)
leg2, = ax3.plot(irx, irz, 'k^', markersize=8)
im3 = ax3.imshow(p, vmin=-lim, vmax=+lim, interpolation='nearest', cmap=plt.cm.RdBu)
div = make_axes_locatable(ax3)
cax = div.append_axes('right', size='5%', pad=0.05)
fig2.colorbar(im3, cax=cax)
ax3.set_title('Time Step (nt) = 0')
ax3.set_xlim(0, nx)
ax3.set_ylim(0, nz)
ax3.set_xlabel('nx')
ax3.set_ylabel('nz')
ax3.legend((leg1, leg2), ('Source', 'Receiver'), loc='upper right', fontsize=10, numpoints=1)

ax4 = plt.subplot(gs2[1])
up41, = ax4.plot(time_values, seis)
up42, = ax4.plot([0], [0], 'r|', markersize=15)
ax4.set_xlim(time_values[0], time_values[-1])
ax4.set_title('Seismogram')
ax4.set_xlabel('Time [s]')
ax4.set_ylabel('Amplitude')
leg3, = ax4.plot(0, 0, 'r--', markersize=1)
leg4, = ax4.plot(0, 0, 'b-', markersize=1)
ax4.legend((leg3, leg4), ('Analytical', 'FD'), loc='upper right', fontsize=10, numpoints=1)
ax4.yaxis.tick_right()
ax4.yaxis.set_label_position('right')
plt.plot(time_values, Gc, 'r--')
plt.show()

In [ ]:
p_results = np.zeros((math.ceil(nt / idisp), nz, nx), dtype=np.float32)
seis_results = np.zeros((math.ceil(nt / idisp), nt), dtype=np.float32)

start = time.perf_counter()
progress_handler = ProgressBarHandler(nt, 'Homogenous 2D Wave Equation Simulation')
for it in range(nt):
    if op == 3:
        for i in range(1, nx - 1):
            d2px[i, :] = (p[i - 1, :] - 2 * p[i, :] + p[i + 1, :]) / dx ** 2
        for j in range(1, nz - 1):
            d2pz[:, j] = (p[:, j - 1] - 2 * p[:, j] + p[:, j + 1]) / dz ** 2

    pnew = 2 * p - pold + (c ** 2) * (dt ** 2) * (d2pz + d2px)
    pnew[isz, isx] = pnew[isz, isx] + src[it] / (dx * dz) * (dt ** 2)
    pold, p = p, pnew
    seis[it] = p[irz, irx]

    if it % idisp == 0:
        p_results[it // idisp] = p
        seis_results[it // idisp] = seis

    progress_handler(it)

simulation_seconds = time.perf_counter() - start
print(f'simulation_seconds={simulation_seconds:.3f}')
print(f'frames={len(p_results)}, frame_shape={p_results.shape[1:]}')

In [ ]:
start = time.perf_counter()
ani = create_animation_homogeneous(locals())
animation_creation_seconds = time.perf_counter() - start
print(f'animation_creation_seconds={animation_creation_seconds:.3f}')

start = time.perf_counter()
html = ani.to_jshtml()
jshtml_render_seconds = time.perf_counter() - start
print(f'jshtml_render_seconds={jshtml_render_seconds:.3f}')
print(f'html_size_mb={len(html) / 1024 / 1024:.2f}')
display(HTML(html))
plt.close()